In [6]:
import pandas as pd
from pathlib import Path

# Find the extracted CSV file in our raw directory
raw_dir = Path("src/gdelt_ingestion/data/raw/")
csv_file = list(raw_dir.glob("*.export.CSV"))[0]

print(f"Reading from: {csv_file}")

# GDELT is tab-separated and has NO headers
df = pd.read_csv(csv_file, sep='\t', header=None)

# Display the shape and the first 5 rows
print(f"Rows: {df.shape[0]}, Columns: {df.shape[1]}")
display(df.head())

# Every row in this table represents a single News Event that happened somewhere in the world in the last 15 minutes. 
# GDELT records 61 specific attributes about every event.

# Column Range,What it represents
# 0,Unique ID: Every event gets a GlobalEventID.,1293255184
# 1 - 4,Time: The date the event happened.,"20250309 (March 9, 2025)"
# 5 - 14,Actor 1: Who did the action?,Row 4 shows BUS COMPANY
# 15 - 24,Actor 2: Who was the action done to?,(Hidden in the ...)
# 25 - 34,The Event: What happened? How important is it?,(Hidden in the ...)
# 51 - 58,Geography: Where did it happen?,"Col 52: Vermilion County, Illinois...Col 56 (Lat): 40.1667Col 57 (Long): -87.7506"
# 59 - 60,Source: Where did GDELT find this?,The URL of the news article.

Reading from: src/gdelt_ingestion/data/raw/20260309144500.export.CSV
Rows: 1304, Columns: 61


,0,1,2,3,4,5,6,7,8,9,...,51,52,53,54,55,56,57,58,59,60
0,1293255184,20250309,202503,2025,2025.189,NaN,NaN,NaN,NaN,NaN,...,3,"Vermilion County, Illinois, United States",US,USIL,NaN,40.166700,-87.7506,1785114,20260309144500,https://www.wsiu.org/state-of-illinois/2026-03...
1,1293255185,20250309,202503,2025,2025.189,NaN,NaN,NaN,NaN,NaN,...,4,"Ayia Napa, Famagusta, Cyprus",CY,CY01,14806,34.991700,34.0000,-2738120,20260309144500,https://metro.co.uk/2026/03/09/safe-travel-cyp...
2,1293255186,20250309,202503,2025,2025.189,NaN,NaN,NaN,NaN,NaN,...,3,"Vermilion County, Illinois, United States",US,USIL,NaN,40.166700,-87.7506,1785114,20260309144500,https://www.wsiu.org/state-of-illinois/2026-03...
3,1293255187,20250309,202503,2025,2025.189,NaN,NaN,NaN,NaN,NaN,...,2,"Illinois, United States",US,USIL,NaN,40.336300,-89.0022,IL,20260309144500,https://www.wsiu.org/state-of-illinois/2026-03...
4,1293255188,20250309,202503,2025,2025.189,BUS,COMPANY,NaN,NaN,NaN,...,1,United States,US,US,NaN,39.828175,-98.5795,US,20260309144500,https://www.proactiveinvestors.co.uk/companies...


In [8]:
# 1. We create a dictionary mapping the raw column numbers to real names
# We are only picking the columns "Risk Analyst" actually needs
gdelt_columns = {
    0: 'global_event_id',
    1: 'event_date',
    5: 'actor1_name',
    15: 'actor2_name',
    26: 'event_code',       # The CAMEO code (e.g., 141 = Protest)
    30: 'goldstein_scale',  # Impact score (-10 to +10)
    34: 'avg_tone',         # Sentiment of the news article (-100 to +100)
    52: 'action_geo_name',  # The city/country string
    53: 'action_geo_country',
    56: 'action_geo_lat',
    57: 'action_geo_long',
    60: 'source_url'        # The link to read the actual news
}

# 2. Rename the columns based on the dictionary
df_renamed = df.rename(columns=gdelt_columns)

# 3. Filter the dataframe to KEEP ONLY the columns that are named
df_clean = df_renamed[list(gdelt_columns.values())]

# 4. Show the transformed, "pretty" data
display(df_clean.head(20))

,global_event_id,event_date,actor1_name,actor2_name,event_code,goldstein_scale,avg_tone,action_geo_name,action_geo_country,action_geo_lat,action_geo_long,source_url
0,1293255184,20250309,NaN,BUS,20,3.0,-3.247808,"Vermilion County, Illinois, United States",US,40.166700,-87.7506,https://www.wsiu.org/state-of-illinois/2026-03...
1,1293255185,20250309,NaN,CVL,43,2.8,-0.223214,"Ayia Napa, Famagusta, Cyprus",CY,34.991700,34.0000,https://metro.co.uk/2026/03/09/safe-travel-cyp...
2,1293255186,20250309,NaN,USABUS,20,3.0,-3.247808,"Vermilion County, Illinois, United States",US,40.166700,-87.7506,https://www.wsiu.org/state-of-illinois/2026-03...
3,1293255187,20250309,NaN,USABUS,20,3.0,-3.247808,"Illinois, United States",US,40.336300,-89.0022,https://www.wsiu.org/state-of-illinois/2026-03...
4,1293255188,20250309,BUS,NOR,111,-2.0,3.736921,United States,US,39.828175,-98.5795,https://www.proactiveinvestors.co.uk/companies...
5,1293255189,20250309,BUS,NOR,111,-2.0,3.736921,Norway,NO,62.000000,10.0000,https://www.proactiveinvestors.co.uk/companies...
6,1293255190,20250309,CVL,NaN,42,1.9,-0.223214,"Ayia Napa, Famagusta, Cyprus",CY,34.991700,34.0000,https://metro.co.uk/2026/03/09/safe-travel-cyp...
7,1293255191,20250309,CVL,IGOEUREEC,42,1.9,-0.223214,"Ayia Napa, Famagusta, Cyprus",CY,34.991700,34.0000,https://metro.co.uk/2026/03/09/safe-travel-cyp...
8,1293255192,20250309,CVL,USA,10,0.0,0.452489,"Iowa, United States",US,42.004600,-93.2140,https://www.kmaland.com/news/nishnabotna-water...
9,1293255193,20250309,GOV,LEG,10,0.0,-1.750973,"Mangaluru, Karnataka, India",IN,12.863900,74.8353,https://news.webindia123.com/news/Articles/Bus...


In [10]:
import pandas as pd

# Complete CAMOE Dictionary
cameo_type_map = {
    "COP": "Police forces",
    "GOV": "Government",
    "INS": "Insurgents",
    "JUD": "Judiciary",
    "MIL": "Military",
    "OPP": "Political Opposition",
    "REB": "Rebels",
    "SEP": "Separatist Rebels",
    "SPY": "State Intelligence",
    "UAF": "Unaligned Armed Forces",
    "AGR": "Agriculture",
    "BUS": "Business",
    "CRM": "Criminal",
    "CVL": "Civilian",
    "DEV": "Development",
    "EDU": "Education",
    "ELI": "Elites",
    "ENV": "Environmental",
    "HLH": "Health",
    "HRI": "Human Rights",
    "LAB": "Labor",
    "LEG": "Legislature",
    "MED": "Media",
    "REF": "Refugees",
    "MOD": "Moderate",
    "RAD": "Radical",
    "AMN": "Amnesty International",
    "IRC": "Red Cross",
    "GRP": "Greenpeace",
    "UNO": "United Nations",
    "PKO": "Peacekeepers",
    "UIS": "Unidentified State Actor",
    "IGO": "Inter-Governmental Organization",
    "IMG": "International Militarized Group",
    "INT": "International/Transnational Generic",
    "MNC": "Multinational Corporation",
    "NGM": "Non-Governmental Movement",
    "NGO": "Non-Governmental Organization",
    "SET": "Settler"
}

# 2. Translator Function
def translate_gdelt_actor(raw_code):
    """Searches the raw GDELT string for any known 3-letter CAMEO codes."""
    if pd.isna(raw_code):
        return "Unknown"
    
    raw_code = str(raw_code).upper()
    found_roles = []
    
    # Check if any of our 3-letter codes exist inside the raw string
    for code, label in cameo_type_map.items():
        if code in raw_code:
            found_roles.append(label)
            
    # If we found matches, join them (e.g., "Government / Police forces")
    if found_roles:
        return " / ".join(found_roles)
    
    # If no 3-letter code was found, just return the original string
    return raw_code

# 3. Apply the transformation to the dataframe
df_clean['actor1_role'] = df_clean['actor1_name'].apply(translate_gdelt_actor)
df_clean['actor2_role'] = df_clean['actor2_name'].apply(translate_gdelt_actor)

# 4. View the transformed data
display(df_clean[['actor1_name', 'actor1_role', 'actor2_name', 'actor2_role']].head(20))

,actor1_name,actor1_role,actor2_name,actor2_role
0,NaN,Unknown,BUS,Business
1,NaN,Unknown,CVL,Civilian
2,NaN,Unknown,USABUS,Business
3,NaN,Unknown,USABUS,Business
4,BUS,Business,NOR,NOR
5,BUS,Business,NOR,NOR
6,CVL,Civilian,NaN,Unknown
7,CVL,Civilian,IGOEUREEC,Inter-Governmental Organization
8,CVL,Civilian,USA,USA
9,GOV,Government,LEG,Legislature


In [9]:
# 1. A dictionary of common CAMEO Actor codes
actor_mapping = {
    'IGOEUREEC': 'European Union',
    'IGOUNO': 'United Nations',
    'USA': 'United States',
    'COP': 'Police Forces',
    'GOV': 'Government',
    'REB': 'Rebels/Insurgents',
    'BUS': 'Business/Corporation',
    'JUD': 'Judiciary/Courts',
    'CVL': 'Civilians'
}

# 2. Apply the mapping to columns
# The `.map()` function looks up the code. 
# If it doesn't find it in the dictionary, `.fillna()` keeps the original code.
df_clean['actor1_name_translated'] = df_clean['actor1_name'].map(actor_mapping).fillna(df_clean['actor1_name'])
df_clean['actor2_name_translated'] = df_clean['actor2_name'].map(actor_mapping).fillna(df_clean['actor2_name'])

# Display newly translated colums
display(df_clean[['actor1_name', 'actor1_name_translated', 'actor2_name', 'actor2_name_translated']].head(20))

,actor1_name,actor1_name_translated,actor2_name,actor2_name_translated
0,NaN,NaN,BUS,Business/Corporation
1,NaN,NaN,CVL,Civilians
2,NaN,NaN,USABUS,USABUS
3,NaN,NaN,USABUS,USABUS
4,BUS,Business/Corporation,NOR,NOR
5,BUS,Business/Corporation,NOR,NOR
6,CVL,Civilians,NaN,NaN
7,CVL,Civilians,IGOEUREEC,European Union
8,CVL,Civilians,USA,United States
9,GOV,Government,LEG,LEG


In [11]:
# 1. Expand dictionaries with Regions, Groups, and some major Countries
cameo_regions = {
    "AFR": "Africa",
    "ASI": "Asia",
    "EUR": "Europe",
    "MEA": "Middle East",
    "SAM": "South America",
    "NAM": "North America"
}

cameo_known_groups = {
    "EEC": "European Union",
    "UNO": "United Nations",
    "IMF": "International Monetary Fund",
    "WBG": "World Bank",
    "WHO": "World Health Organization"
}

cameo_countries = {
    "USA": "United States",
    "RUS": "Russia",
    "CHN": "China",
    "GBR": "United Kingdom",
    "FRA": "France",
    "DEU": "Germany",
    "UKR": "Ukraine",
    "ISR": "Israel",
    "PSE": "Palestine",
    "CHE": "Switzerland" 
}

# Combine original type map with the new ones into one Mega-Dictionary
# (Assuming cameo_type_map from the previous cell is still in memory)
master_cameo_map = {**cameo_type_map, **cameo_regions, **cameo_known_groups, **cameo_countries}

# 2. The Chunking Function
def fully_translate_cameo(raw_code):
    """Slices a GDELT code into 3-letter chunks and translates each part."""
    if pd.isna(raw_code):
        return "Unknown"
    
    raw_code = str(raw_code).upper()
    
    # Slice the string into blocks of 3
    # e.g., 'IGOEUREEC' becomes ['IGO', 'EUR', 'EEC']
    chunks = [raw_code[i:i+3] for i in range(0, len(raw_code), 3)]
    
    translated_parts = []
    for chunk in chunks:
        # Lookup the chunk. If we don't have it in our dict, just return the raw 3 letters
        translation = master_cameo_map.get(chunk, chunk)
        translated_parts.append(translation)
        
    return " - ".join(translated_parts)

# 3. Apply the ultimate transformation
df_clean['actor1_full_translation'] = df_clean['actor1_name'].apply(fully_translate_cameo)

# 4. Check the results (filtering out the "Unknowns")
mask = df_clean['actor1_name'].notna()
display(df_clean[mask][['actor1_name', 'actor1_full_translation']].head(20))

,actor1_name,actor1_full_translation
4,BUS,Business
5,BUS,Business
6,CVL,Civilian
7,CVL,Civilian
8,CVL,Civilian
9,GOV,Government
10,IGOEUREEC,Inter-Governmental Organization - Europe - Eur...
11,USA,United States
12,USA,United States
13,USA,United States
